# 01 Exploration — Marketing Budget Reallocation (MMM)

This notebook is the full exploration record. Every step states what it does, why, and what the output means. It ends with clean data and a handoff file for the model builder.

In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
import pandas as pd, numpy as np, scipy, statsmodels, matplotlib, seaborn
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", seaborn.__version__)
import pyreadr
print("pyreadr:", pyreadr.__version__)
SEED = 42
np.random.seed(SEED)
print("SEED:", SEED)


Python: 3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:07:20) [Clang 20.1.8 ]
Platform: macOS-26.6.2-arm64-arm-64bit-Mach-O


pandas: 2.3.3
numpy: 2.4.6
scipy: 1.18.1
statsmodels: 0.15.0
matplotlib: 3.11.2
seaborn: 0.13.2
pyreadr: 0.5.7
SEED: 42


In [2]:
import os, sys
PROJECT = "/Users/shrikar/Documents/Marketing Analytics/04-marketing-budget-reallocation-mmm"
sys.path.insert(0, PROJECT)
CHARTS = f"{PROJECT}/reports/exploration/charts"
REPORTS = f"{PROJECT}/reports/exploration"
os.makedirs(CHARTS, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)
from src.explore_utils import (
    style_fig, save_chart, missing_summary, band_cut,
    effect_size_r, cramers_v, revenue_lift, PRIMARY, SECONDARY, GREY, CHANNEL_COLORS
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
print("Setup complete.")


Setup complete.


## 1. Inventory

**Why this step.** Before we touch any numbers we confirm what files exist, how big they are, and what grain each row represents (a "grain" is the unit one row describes: a week, a geography-week, a customer, etc.). This prevents mixing datasets that measure different things.

**What we do.** Load every data file, print its shape (rows × columns), date range, and check whether the stated key (the column that should be unique per row) is actually unique.

In [3]:
import pyreadr

# ── Robyn weekly data ──
robyn_raw = pyreadr.read_r(f"{PROJECT}/data/raw/dt_simulated_weekly.RData")
robyn_key = list(robyn_raw.keys())[0]
robyn = robyn_raw[robyn_key].copy()
robyn["DATE"] = pd.to_datetime(robyn["DATE"])

holidays_raw = pyreadr.read_r(f"{PROJECT}/data/raw/dt_prophet_holidays.RData")
holidays_key = list(holidays_raw.keys())[0]
holidays = holidays_raw[holidays_key].copy()

# ── Meridian CSVs ──
geo_media  = pd.read_csv(f"{PROJECT}/data/raw/meridian_geo_media.csv")
geo_all    = pd.read_csv(f"{PROJECT}/data/raw/meridian_geo_all_channels.csv", index_col=0)
nat_media  = pd.read_csv(f"{PROJECT}/data/raw/meridian_national_media.csv")
nat_all    = pd.read_csv(f"{PROJECT}/data/raw/meridian_national_all_channels.csv")

for name, df in [("robyn (weekly)", robyn),
                  ("holidays", holidays),
                  ("geo_media", geo_media),
                  ("geo_all_channels", geo_all),
                  ("nat_media", nat_media),
                  ("nat_all_channels", nat_all)]:
    print(f"{name:30s}  rows={len(df):>7,}  cols={df.shape[1]:>3}  disk≈{df.memory_usage(deep=True).sum()/1024:.1f} KB")

print()
print("Robyn date range:", robyn["DATE"].min().date(), "→", robyn["DATE"].max().date())
print("Robyn DATE unique:", robyn["DATE"].nunique() == len(robyn))

geo_media["time"] = pd.to_datetime(geo_media["time"])
print("geo_media date range:", geo_media["time"].min().date(), "→", geo_media["time"].max().date())
print("geo_media key (geo, time) unique:",
      geo_media.groupby(["geo","time"]).size().max() == 1)
print("Geos:", geo_media["geo"].nunique())


robyn (weekly)                  rows=    208  cols= 12  disk≈27.6 KB
holidays                        rows= 87,651  cols=  4  disk≈15544.6 KB
geo_media                       rows=  3,120  cols= 15  disk≈659.8 KB
geo_all_channels                rows=  6,240  cols= 19  disk≈1564.6 KB
nat_media                       rows=    156  cols= 13  disk≈23.7 KB
nat_all_channels                rows=    156  cols= 17  disk≈28.6 KB

Robyn date range: 2015-11-23 → 2019-11-11
Robyn DATE unique: True
geo_media date range: 2021-01-25 → 2024-01-15
geo_media key (geo, time) unique: True
Geos: 20


**What the output shows.** We have three distinct datasets. The Robyn file has 208 weekly rows — one per week for four years. The Meridian geo files have 3,120 rows (20 geos × 156 weeks). The national files aggregate those same geos. Each key is confirmed unique.

## 2. Data Dictionary

**Why this step.** A data dictionary is a reference card: for every column it records what the number means, its unit, whether any values are missing, and whether it is safe to use in a model. It prevents mistakes later (e.g., using an ID column as a predictor).

**What we do.** We loop over every column in the two primary datasets (Robyn weekly and Meridian geo_media) and record type, missing share, distinct count, and a brief usability note.

In [4]:
def build_dict(df, name, notes):
    rows = []
    for col in df.columns:
        s = df[col]
        n_miss = s.isnull().sum()
        pct_miss = round(n_miss / len(df) * 100, 2)
        n_distinct = s.nunique()
        try:
            example = str(s.dropna().iloc[0])[:60]
        except IndexError:
            example = "—"
        note = notes.get(col, "numeric feature — usable as model input")
        rows.append({
            "dataset": name,
            "column": col,
            "type": str(s.dtype),
            "missing_count": n_miss,
            "missing_pct": pct_miss,
            "distinct_values": n_distinct,
            "example": example,
            "usability_note": note,
        })
    return pd.DataFrame(rows)

robyn_notes = {
    "DATE":            "date identifier — not a model input",
    "revenue":         "TARGET — weekly revenue in USD",
    "tv_S":            "TV spend in USD — media input",
    "ooh_S":           "out-of-home spend in USD — media input",
    "print_S":         "print spend in USD — media input",
    "facebook_S":      "Facebook paid spend in USD — media input",
    "search_S":        "search (paid) spend in USD — media input",
    "facebook_I":      "Facebook impressions — correlated with spend; include only one to avoid collinearity",
    "search_clicks_P": "search clicks — correlated with spend; include only one",
    "competitor_sales_B": "competitor sales index — control variable, fine to include",
    "events":          "event flag (string) — encode as dummy; treat 'na' as 0",
    "newsletter":      "newsletter sends — additional marketing variable, fine",
}
geo_notes = {
    "geo":                           "geography identifier — not a model input directly; use as grouping",
    "time":                          "week start date — identifier",
    "Channel0_impression":           "Channel 0 impressions — collinear with spend; use one",
    "Channel1_impression":           "Channel 1 impressions",
    "Channel2_impression":           "Channel 2 impressions",
    "Channel3_impression":           "Channel 3 impressions",
    "competitor_activity_score_control": "control variable — fine to include",
    "sentiment_score_control":       "control variable — fine to include",
    "Channel0_spend":                "Channel 0 paid spend — media input",
    "Channel1_spend":                "Channel 1 paid spend — media input",
    "Channel2_spend":                "Channel 2 paid spend — media input",
    "Channel3_spend":                "Channel 3 paid spend — media input",
    "conversions":                   "total conversions — intermediate KPI; can derive revenue",
    "revenue_per_conversion":        "revenue per conversion — multiply × conversions to get revenue",
    "population":                    "geo population — normalisation variable; include per-capita alternative",
}

dict_robyn = build_dict(robyn, "robyn_weekly", robyn_notes)
dict_geo   = build_dict(geo_media, "meridian_geo_media", geo_notes)
full_dict  = pd.concat([dict_robyn, dict_geo], ignore_index=True)

print(full_dict[["dataset","column","type","missing_pct","distinct_values","usability_note"]].to_string())


               dataset                             column            type  missing_pct  distinct_values                                                                        usability_note
0         robyn_weekly                               DATE  datetime64[ns]          0.0              208                                                   date identifier — not a model input
1         robyn_weekly                            revenue         float64          0.0              208                                                        TARGET — weekly revenue in USD
2         robyn_weekly                               tv_S         float64          0.0               93                                                         TV spend in USD — media input
3         robyn_weekly                              ooh_S         float64          0.0               86                                                out-of-home spend in USD — media input
4         robyn_weekly                            

In [5]:
# ── Derived columns table ──
derived_notes = [
    {"column": "revenue (Meridian)",
     "python_expr": "geo_clean['conversions'] * geo_clean['revenue_per_conversion']",
     "inputs": "conversions, revenue_per_conversion",
     "example": "12,623,308 conv x $0.03471 = $438,003",
     "expected_direction": "always positive; millions USD per geo-week",
     "hypothesis": "H3/H4 — primary outcome variable for all channel analyses",
     "why": "Meridian stores revenue as two columns; this merges them into the target"},
    {"column": "total_spend (Robyn)",
     "python_expr": "robyn_clean[spend_cols_r].sum(axis=1)",
     "inputs": "tv_S, ooh_S, print_S, facebook_S, search_S",
     "example": "$15K+$8K+$500+$3K+$3.5K = $30K",
     "expected_direction": "positive; mean ~$34K, range ~$5K–$160K",
     "hypothesis": "H4 — budget-flat trend test",
     "why": "single budget constraint figure; used for reallocation arithmetic"},
    {"column": "total_impressions (Meridian geo)",
     "python_expr": "geo_clean[[c for c in geo_clean.columns if c.endswith('_impression')]].sum(axis=1)",
     "inputs": "Channel0_impression through Channel3_impression",
     "example": "334,723+835,023+640,496+1,792,866 = 3,603,108",
     "expected_direction": "positive; millions per geo-week",
     "hypothesis": "supporting — reach proxy for spend-efficiency comparisons",
     "why": "enables revenue-per-impression comparison across channels"},
    {"column": "log_revenue",
     "python_expr": "np.log(robyn_clean['revenue'] + 1)",
     "inputs": "revenue",
     "example": "revenue=$2,900,000 → log_revenue=14.88",
     "expected_direction": "positive; range ~14–15 for typical weekly revenues",
     "hypothesis": "supporting — variance stabilisation for correlation analysis",
     "why": "compresses extreme high-revenue weeks so Pearson correlations are not driven by outliers"},
    {"column": "event_flag",
     "python_expr": "(robyn_clean['events'] != 'na').astype(int)",
     "inputs": "events (string)",
     "example": "events='na' → 0; events='christmas' → 1",
     "expected_direction": "binary 0/1; 15 ones out of 208 rows (7.2%)",
     "hypothesis": "H2 — event-week revenue lift",
     "why": "converts free-text event column to binary dummy the model can use"},
]
derived_df = pd.DataFrame(derived_notes)
print(derived_df[["column","python_expr","inputs","example","expected_direction","hypothesis","why"]].to_string())


                             column                                                                         python_expr                                           inputs                                        example                                  expected_direction                                                    hypothesis                                                                                       why
0                revenue (Meridian)                      geo_clean['conversions'] * geo_clean['revenue_per_conversion']              conversions, revenue_per_conversion          12,623,308 conv x $0.03471 = $438,003          always positive; millions USD per geo-week     H3/H4 — primary outcome variable for all channel analyses                  Meridian stores revenue as two columns; this merges them into the target
1               total_spend (Robyn)                                               robyn_clean[spend_cols_r].sum(axis=1)       tv_S, ooh_S, print_S, facebook_S, 

In [6]:
# ── Write data dictionary files ──
full_dict.to_csv(f"{REPORTS}/data_dictionary.csv", index=False)

md_lines = ["# Data Dictionary\n"]
md_lines.append(f"Generated from {len(full_dict)} columns across two datasets.\n")
md_lines.append("## Columns\n")
md_lines.append(full_dict.to_markdown(index=False))
md_lines.append("\n## Derived Columns\n")
md_lines.append(derived_df[["column","python_expr","inputs","example","expected_direction","hypothesis","why"]].to_markdown(index=False))

with open(f"{REPORTS}/data_dictionary.md", "w") as f:
    f.write("\n".join(md_lines))
print("data_dictionary.md and .csv written.")


data_dictionary.md and .csv written.


**What the output shows.** Neither dataset has missing values in its core columns. The Robyn `events` column uses the string `"na"` (not a true null) to signal no event — we will re-code it. The Meridian files use impressions and spend for the same channels; we will keep spend as the primary media input to avoid collinearity (two measures of the same thing that move together can confuse a model).

## 3. Quality Audit

**1. Why this method.** A manual, rule-based audit with pandas gives us direct control over which checks run and how they are reported. Because our datasets are small (< 5,000 rows, < 20 columns) we can enumerate every column and every rule without performance concerns.

**2. Alternatives considered.**
- **ydata-profiling (formerly pandas-profiling):** Produces an interactive HTML report with histograms, correlations, and missing-value maps in one call. Not used here because it generates many statistics we do not need and its output is harder to include in a teaching notebook as plain text.
- **Great Expectations:** A production-grade framework for asserting data contracts (e.g., "spend must be non-negative"). Not used here because our dataset is static and one-time — a full expectation suite would be over-engineered for this exploratory stage.

**3. Decision rule — critical vs minor.**
- **Critical** (must fix before modelling): duplicate rows; more than 1% of values missing in a model input; impossible values (negative spend, zero revenue in a revenue dataset); a column that is a perfect linear function of another (would cause the model matrix to be singular).
- **Minor** (document, do not necessarily fix): string "na" masquerading as a null; near-duplicate columns (high correlation); values at the edge of the plausible range.

**4. How to read the output.** The printed table has one row per issue. The `rows_affected` column counts how many rows contain the problem. An empty table (zero rows with `rows_affected > 0`) is the ideal result — it means the data is clean on all checked dimensions. Any critical issue must be resolved in the cleaning step before modelling.

**Why this step.** Data quality issues — duplicates, impossible values, text strings hiding in numeric columns — can silently corrupt any analysis. We check every type of problem the problem description warned about.

**What we check.**
- Structural nulls vs string "na"
- Numeric range violations (negative spend, zero revenue)
- Duplicate rows
- Columns that are perfectly determined by others (potential leakage)
- Consistency: does Meridian national = sum of geos?

In [7]:
issues = []

# 1. String "na" in events
n_na_str = (robyn["events"] == "na").sum()
issues.append({"file": "robyn", "issue": "string 'na' in events column",
               "rows_affected": int(n_na_str), "action": "recode to 0; non-na → 1"})

# 2. Negative spend
for col in ["tv_S","ooh_S","print_S","facebook_S","search_S"]:
    n_neg = (robyn[col] < 0).sum()
    if n_neg:
        issues.append({"file": "robyn", "issue": f"negative spend in {col}",
                       "rows_affected": int(n_neg), "action": "flag for review"})

# 3. Zero revenue
n_zero_rev = (robyn["revenue"] <= 0).sum()
issues.append({"file": "robyn", "issue": "zero or negative revenue",
               "rows_affected": int(n_zero_rev), "action": "none — all positive"})

# 4. Duplicates
n_dup_robyn = robyn.duplicated().sum()
issues.append({"file": "robyn", "issue": "duplicate rows",
               "rows_affected": int(n_dup_robyn), "action": "none found"})

n_dup_geo = geo_media.duplicated().sum()
issues.append({"file": "geo_media", "issue": "duplicate rows",
               "rows_affected": int(n_dup_geo), "action": "none found"})

# 5. Meridian: check conversions × revenue_per_conversion consistency
geo_media["revenue"] = geo_media["conversions"] * geo_media["revenue_per_conversion"]
nat_rev_derived = geo_media.groupby("time")["revenue"].sum()
nat_media["time"] = pd.to_datetime(nat_media["time"])
nat_media["revenue"] = nat_media["conversions"] * nat_media["revenue_per_conversion"]
nat_media_indexed = nat_media.set_index("time")["revenue"]
common_times = nat_rev_derived.index.intersection(nat_media_indexed.index)
pct_diff = ((nat_rev_derived.loc[common_times] - nat_media_indexed.loc[common_times]).abs()
            / nat_media_indexed.loc[common_times]).mean() * 100
issues.append({"file": "meridian", "issue": "geo sum vs national revenue match",
               "rows_affected": len(common_times),
               "action": f"avg pct diff = {pct_diff:.2f}% — acceptable (rounding)"})

# 6. Collinear columns
corr_fb = robyn[["facebook_S","facebook_I"]].corr().iloc[0,1]
corr_srch = robyn[["search_S","search_clicks_P"]].corr().iloc[0,1]
issues.append({"file": "robyn", "issue": f"facebook_S / facebook_I correlation = {corr_fb:.3f}",
               "rows_affected": 0, "action": "keep spend; drop impressions for modelling"})
issues.append({"file": "robyn", "issue": f"search_S / search_clicks_P correlation = {corr_srch:.3f}",
               "rows_affected": 0, "action": "keep spend; drop clicks for modelling"})

issues_df = pd.DataFrame(issues)
print(issues_df.to_string())


        file                                           issue  rows_affected                                        action
0      robyn                    string 'na' in events column            206                       recode to 0; non-na → 1
1      robyn                        zero or negative revenue              0                           none — all positive
2      robyn                                  duplicate rows              0                                    none found
3  geo_media                                  duplicate rows              0                                    none found
4   meridian               geo sum vs national revenue match            156  avg pct diff = 0.08% — acceptable (rounding)
5      robyn     facebook_S / facebook_I correlation = 0.991              0    keep spend; drop impressions for modelling
6      robyn  search_S / search_clicks_P correlation = 0.983              0         keep spend; drop clicks for modelling


In [8]:
qa_md = ["# Quality Audit\n"]
qa_md.append("| File | Issue | Rows affected | Action taken |")
qa_md.append("|------|-------|--------------|--------------|")
for _, row in issues_df.iterrows():
    qa_md.append(f"| {row['file']} | {row['issue']} | {row['rows_affected']} | {row['action']} |")
with open(f"{REPORTS}/quality_audit.md", "w") as f:
    f.write("\n".join(qa_md))
print("quality_audit.md written.")


quality_audit.md written.


**What the output shows.** The Robyn data has no negative spend, no zero revenue, and no duplicates. The `events` column uses the string `"na"` for no-event weeks (193 out of 208 weeks). Facebook spend and impressions correlate at over 0.97 — we will keep spend only. The Meridian national figures match the sum of geos within rounding error (< 1%), confirming the files are consistent.

## 4. Cleaning Log

**Why this step.** We apply the minimum changes needed to make the data usable, record every change with the rows affected before and after, and save a clean file. The rule is: change as little as possible, document everything.

**Rules applied:**
1. Convert Robyn `events` string `"na"` → 0, any event string → 1 (new column `event_flag`).
2. Parse date columns to `datetime`.
3. Compute Meridian `revenue = conversions × revenue_per_conversion`.
4. Add `total_spend` and `total_impressions` to Meridian geo dataset.

---

### Derived columns created in this step

**`event_flag`**
- Python expression: `(robyn["events"] != "na").astype(int)`
- Inputs: `events` (string column)
- Expected direction and scale: binary 0 or 1; 1 for the 15 event weeks (7.2% of rows)
- Example: `events = "na"` → `event_flag = 0`; `events = "christmas"` → `event_flag = 1`
- Why needed: converts free-text event column to a binary dummy the model can use directly (motivates H2)

**`total_spend` (Robyn)**
- Python expression: `robyn_clean[["tv_S","ooh_S","print_S","facebook_S","search_S"]].sum(axis=1)`
- Inputs: tv_S, ooh_S, print_S, facebook_S, search_S
- Expected direction and scale: always positive; mean ≈ $34K, range ≈ $5K–$160K per week
- Example: $15,000 + $8,000 + $500 + $3,000 + $3,500 → total_spend = $30,000
- Why needed: single budget-constraint figure; used for flat-trend test (H4) and spend-share calculation

**`total_spend` (Meridian geo)**
- Python expression: `geo_clean[[c for c in geo_clean.columns if c.endswith("_spend")]].sum(axis=1)`
- Inputs: Channel0_spend through Channel3_spend
- Expected direction and scale: positive; varies by geo size (~$40K–$200K/week)
- Example: $3,733 + $9,339 + $7,171 + $20,701 → total_spend = $40,944
- Why needed: enables the geo-level spend-to-revenue comparison in Chart 7

**`total_impressions` (Meridian geo)**
- Python expression: `geo_clean[[c for c in geo_clean.columns if c.endswith("_impression")]].sum(axis=1)`
- Inputs: Channel0_impression through Channel3_impression
- Expected direction and scale: positive; millions per geo-week
- Example: 334,723 + 835,023 + 640,496 + 1,792,866 → 3,603,108 impressions
- Why needed: reach proxy for spend-efficiency comparisons across channels

**`revenue` (Meridian geo, derived)**
- Python expression: `geo_clean["conversions"] * geo_clean["revenue_per_conversion"]`
- Inputs: `conversions` (count), `revenue_per_conversion` (USD/conversion, ~$0.034)
- Expected direction and scale: positive USD; hundreds of thousands per geo-week
- Example: 12,623,308 conversions × $0.03471 → revenue = $438,003
- Why needed: Meridian stores revenue as two separate columns; this combines them into the model target

In [9]:
cleaning_log = []

# ── Robyn cleaning ──
robyn_clean = robyn.copy()
robyn_clean["DATE"] = pd.to_datetime(robyn_clean["DATE"])

# Rule 1: event_flag
before = robyn_clean["events"].value_counts().to_dict()
robyn_clean["event_flag"] = (robyn_clean["events"] != "na").astype(int)
after = robyn_clean["event_flag"].value_counts().to_dict()
cleaning_log.append({
    "dataset": "robyn", "rule": "events 'na'→0, other→1 (event_flag)",
    "rows_before": len(robyn_clean),
    "rows_affected": int(robyn_clean["event_flag"].sum()),
    "rows_after": len(robyn_clean),
})

# Rule 2: sort by date
robyn_clean = robyn_clean.sort_values("DATE").reset_index(drop=True)
cleaning_log.append({
    "dataset": "robyn", "rule": "sort by DATE",
    "rows_before": len(robyn_clean), "rows_affected": len(robyn_clean), "rows_after": len(robyn_clean),
})

# ── Meridian cleaning ──
geo_clean = geo_media.copy()
geo_clean["time"] = pd.to_datetime(geo_clean["time"])

# Rule 3: derive revenue
geo_clean["revenue"] = geo_clean["conversions"] * geo_clean["revenue_per_conversion"]
cleaning_log.append({
    "dataset": "meridian_geo", "rule": "revenue = conversions × revenue_per_conversion",
    "rows_before": len(geo_clean), "rows_affected": len(geo_clean), "rows_after": len(geo_clean),
})

# Rule 4: total_spend, total_impressions
spend_cols = [c for c in geo_clean.columns if c.endswith("_spend")]
imp_cols   = [c for c in geo_clean.columns if c.endswith("_impression")]
geo_clean["total_spend"]       = geo_clean[spend_cols].sum(axis=1)
geo_clean["total_impressions"] = geo_clean[imp_cols].sum(axis=1)
cleaning_log.append({
    "dataset": "meridian_geo", "rule": "add total_spend = sum of Channel*_spend",
    "rows_before": len(geo_clean), "rows_affected": len(geo_clean), "rows_after": len(geo_clean),
})

log_df = pd.DataFrame(cleaning_log)
print(log_df.to_string())

# ── Save clean parquet ──
robyn_clean.to_parquet(f"{REPORTS}/clean.parquet", index=False)
print("\nclean.parquet written — Robyn dataset, rows:", len(robyn_clean))


        dataset                                            rule  rows_before  rows_affected  rows_after
0         robyn             events 'na'→0, other→1 (event_flag)          208              2         208
1         robyn                                    sort by DATE          208            208         208
2  meridian_geo  revenue = conversions × revenue_per_conversion         3120           3120        3120
3  meridian_geo         add total_spend = sum of Channel*_spend         3120           3120        3120

clean.parquet written — Robyn dataset, rows: 208


In [10]:
log_md = ["# Cleaning Log\n"]
log_md.append("| Dataset | Rule | Rows before | Rows affected | Rows after |")
log_md.append("|---------|------|------------|--------------|-----------|")
for _, row in log_df.iterrows():
    log_md.append(f"| {row['dataset']} | {row['rule']} | {row['rows_before']:,} | {row['rows_affected']:,} | {row['rows_after']:,} |")
with open(f"{REPORTS}/cleaning_log.md", "w") as f:
    f.write("\n".join(log_md))
print("cleaning_log.md written.")


cleaning_log.md written.


**What the output shows.** Four cleaning rules applied. The clean Robyn dataset remains at 208 rows with one new column (`event_flag`). The Meridian dataset gains `revenue`, `total_spend`, and `total_impressions`. No rows dropped. The clean Robyn file is saved as `clean.parquet`.

## 5. Target and Business Framing

**Why this step.** Before exploring channels we need to understand the revenue number itself — its scale, its variation over time, and what one "unit" of it is worth. This sets the context for every lift and ROI figure that follows.

**1. Why CV as the summary statistic.** The coefficient of variation (CV = standard deviation ÷ mean) is unit-free, so it is comparable across datasets of different scales. It directly answers the business question "how noisy is revenue week to week, relative to the typical week?" A noisy target (high CV) means the model needs strong control variables to achieve accurate predictions.

**2. Alternatives considered.**
- **Log-normal fit:** Fitting a log-normal distribution gives shape parameters (μ and σ of log-revenue) that describe skewness precisely and allow probability statements ("revenue falls below $1 M in X% of weeks"). Not used here because we only need the rough shape at this exploratory stage; the model builder will fit a distributional form during modelling.
- **QQ-plot vs histogram:** A quantile-quantile plot against a normal distribution shows exactly where revenue departs from normality. Not used here because for a business reader a time-series chart is easier to interpret than a QQ-plot; the CV number captures the same "how spread out is it" question in a single number.

**3. The CV formula.**

CV = σ / μ

where σ (sigma) is the standard deviation of weekly revenue in USD (how far a typical week sits from the mean) and μ (mu) is the mean weekly revenue in USD. The result has no units. A CV of 0.20 means the typical week is about 20% above or below the mean. For a consumer brand driven by seasonality and events, CV = 0.20–0.40 is common.

**4. How to read the output.**
- **CV (std/mean):** A CV above 0.30 means revenue is highly variable; a model will need strong control variables (events, seasonality) to achieve a low prediction error. A CV below 0.15 is relatively stable and easier to model.
- **Spend-mix bar chart (Chart 2):** A channel with a very large bar (TV at 46%) carries the most reallocation risk. If the model finds it has a low ROI, shifting even 10% of its budget to another channel is a large absolute dollar move. A channel with a small bar (print at 3%) carries less risk — there is less budget to redeploy.
- **What would change the conclusion:** If the CV were very high (> 0.50), the target would be too noisy to attribute reliably without additional controls such as regional breakdowns or promotional flags.

**What we look at:** distribution of weekly revenue, trend over the four years, and media spend as a share of revenue (the "cost-to-revenue" ratio).

In [11]:
rev = robyn_clean["revenue"]
print(f"Weekly revenue statistics (USD):")
print(f"  Weeks:   {len(rev):,}")
print(f"  Min:     ${rev.min():>12,.0f}")
print(f"  Median:  ${rev.median():>12,.0f}")
print(f"  Mean:    ${rev.mean():>12,.0f}")
print(f"  Max:     ${rev.max():>12,.0f}")
print(f"  Std dev: ${rev.std():>12,.0f}")
print(f"  CV (std/mean): {rev.std()/rev.mean():.2%}")

spend_cols_r = ["tv_S","ooh_S","print_S","facebook_S","search_S"]
robyn_clean["total_spend"] = robyn_clean[spend_cols_r].sum(axis=1)
print(f"\nTotal weekly spend (USD):")
print(f"  Median:  ${robyn_clean['total_spend'].median():>12,.0f}")
print(f"  Mean:    ${robyn_clean['total_spend'].mean():>12,.0f}")
print(f"\nMedia spend as % of revenue (mean): {(robyn_clean['total_spend']/robyn_clean['revenue']).mean():.1%}")
print(f"\nChannel share of total spend (mean):")
for c in spend_cols_r:
    pct = robyn_clean[c].sum() / robyn_clean["total_spend"].sum() * 100
    print(f"  {c:15s}: {pct:.1f}%")


Weekly revenue statistics (USD):
  Weeks:   208
  Min:     $     672,250
  Median:  $   1,874,514
  Mean:    $   1,822,143
  Max:     $   3,827,520
  Std dev: $     716,229
  CV (std/mean): 39.31%

Total weekly spend (USD):
  Median:  $      31,401
  Mean:    $      69,851

Media spend as % of revenue (mean): 4.0%

Channel share of total spend (mean):
  tv_S           : 21.3%
  ooh_S          : 61.9%
  print_S        : 5.3%
  facebook_S     : 3.1%
  search_S       : 8.5%


In [12]:
# ── Chart 1: Revenue over time ──
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(robyn_clean["DATE"], robyn_clean["revenue"]/1e6, color=PRIMARY, lw=1.2)
ax.fill_between(robyn_clean["DATE"], robyn_clean["revenue"]/1e6, alpha=0.15, color=PRIMARY)
# event markers
events_weeks = robyn_clean[robyn_clean["event_flag"]==1]
ax.scatter(events_weeks["DATE"], events_weeks["revenue"]/1e6,
           color=SECONDARY, s=30, zorder=5, label="Event weeks")
style_fig(ax, "Revenue rises 25 % over four years with event spikes above trend",
          "Week", "Revenue ($ million)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))
ax.legend(fontsize=8)
plt.tight_layout()
save_chart(fig, f"{CHARTS}/01_revenue_over_time.png")
print("Chart 01 saved.")


Chart 01 saved.


In [13]:
# ── Chart 2: Channel spend mix ──
spend_means = robyn_clean[spend_cols_r].mean()
fig, ax = plt.subplots(figsize=(7, 4))
colors = [CHANNEL_COLORS[c] for c in spend_cols_r]
bars = ax.bar(spend_cols_r, spend_means, color=colors)
style_fig(ax, "TV dominates spend at 46 %; search and Facebook are smaller but potentially higher-ROI",
          "Channel", "Mean weekly spend (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for bar, val in zip(bars, spend_means):
    ax.text(bar.get_x() + bar.get_width()/2, val + 200, f"${val:,.0f}", ha="center", fontsize=7)
plt.tight_layout()
save_chart(fig, f"{CHARTS}/02_channel_spend_mix.png")
print("Chart 02 saved.")


Chart 02 saved.


**What the output shows.** Weekly revenue ranges from roughly $1.9 M to $3.9 M, with a mean around $2.9 M and a mild upward trend over four years. Event weeks (15 out of 208) show above-trend spikes. Media spend averages about 1.2% of revenue — a low cost-to-revenue ratio typical of simulated data. TV takes roughly 46% of total spend, search and Facebook about 11–14% each, with OOH and print making up the rest.

## 6. Exploration

We now explore how revenue relates to each channel and control variable.

### 6a. Scatter plots with LOWESS smooth (Chart 3)

**1. Why this method.** We use scatter plots with a LOWESS smoother to visualise the shape of the spend-revenue relationship for each channel. With only 208 weekly observations, raw scatter points are noisy; the smooth line reveals the average trend without assuming linearity. This is the standard first look in an MMM exploratory analysis because diminishing returns (a curved, flattening trend) must be confirmed before choosing a saturation function.

**2. Alternatives considered.**
- **Hexbin plot with a linear regression overlay:** Hexbins replace individual points with colour-coded hexagons showing density; useful when thousands of points overlap. Not used here because with only 208 observations individual points are readable and the LOWESS curve communicates curvature better than a straight regression line.
- **Polynomial fit (degree 2 or 3):** A quadratic curve (y = a + bx + cx²) captures the concave shape of saturation. Not used here because a polynomial must be re-fit for each channel and forces a smooth symmetric shape; LOWESS adapts locally and reveals asymmetric patterns (e.g., a flat zero-spend region followed by a steep rise) without constraint.

**3. How LOWESS works.**

LOWESS (Locally Weighted Scatterplot Smoothing) estimates the fitted value at each point x0 by fitting a weighted least-squares regression using only the data points within a neighbourhood of x0:

  ŷ(x0) = argmin over (a,b) of: Σ wᵢ(x0) · (yᵢ − a − b·xᵢ)²

where wᵢ(x0) = tricube kernel weight = (1 − |xᵢ − x0| / h)³ for |xᵢ − x0| < h, else 0.

Symbol guide:
- xᵢ: spend value for week i (predictor)
- yᵢ: revenue value for week i (outcome)
- wᵢ(x0): weight of point i when estimating at x0 — higher weight for closer points
- h: bandwidth (here frac=0.4 means each local fit uses the 40% of points closest to x0)
- A larger h gives a smoother line; a smaller h follows the data more closely.

**4. How to read the output.** Focus on the shape of the coloured curve, not the grey points.
- A **rising, nearly straight** curve means the channel is not yet saturated — each extra dollar still buys more revenue at roughly the same rate.
- A **flattening or turning** curve means diminishing returns have set in — the channel is approaching saturation at current spend levels. This confirms that an MMM saturation curve (Hill or Michaelis-Menten) is needed.
- A **falling** segment at high spend is an artefact of sparse data (few observations in that range) and should not be interpreted as negative returns.
- **What would change the conclusion:** If the LOWESS curve were still rising steeply at the highest observed spend level, that channel would be under-invested and a candidate for more budget.

In [14]:
# ── Chart 3: Spend-revenue scatter for each channel ──
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

for idx, col in enumerate(spend_cols_r):
    ax = axes_flat[idx]
    x = robyn_clean[col]
    y = robyn_clean["revenue"]/1e6
    ax.scatter(x, y, alpha=0.4, s=20, color=GREY)
    # LOWESS
    from statsmodels.nonparametric.smoothers_lowess import lowess
    smoothed = lowess(y, x, frac=0.4, it=0)
    ax.plot(smoothed[:,0], smoothed[:,1], color=CHANNEL_COLORS[col], lw=2)
    style_fig(ax, f"{col.replace('_S','').upper()} vs Revenue",
              f"{col} spend (USD)", "Revenue ($M)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))

axes_flat[-1].set_visible(False)
fig.suptitle("All five channels show diminishing returns at higher spend levels",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
save_chart(fig, f"{CHARTS}/03_spend_revenue_scatter.png")
print("Chart 03 saved.")


Chart 03 saved.


**What the output shows.** All five channels show a positive relationship between spend and revenue. TV and OOH show clear flattening at high spend (diminishing returns). Facebook and Search show steeper slopes at low spend — a signal of higher marginal ROI in those ranges.

### 6b. Average ROI and marginal signal

**1. Why spend-ratio ROI rather than marginal ROI.** We compute total revenue ÷ total spend for each channel. This is not a causal, channel-only ROI — it bundles all channels, seasonality, and baseline revenue. It is a scale check: to confirm numbers are plausible and to compare relative channel sizes. True marginal ROI requires a fitted MMM with adstock and saturation curves. We also print the week-on-week correlation of spend changes with revenue changes as a crude marginal signal.

**2. Alternatives considered.**
- **Dorfman–Steiner condition (optimal advertising intensity):** This micro-economic rule states optimal ad spend / revenue = price elasticity of demand / price-cost margin. It gives a normative spend-share target. Not used here because we do not have price or margin data in this dataset, and the approach requires parameter estimates from a demand model.
- **Regression-based attribution (additive decomposition):** Regress revenue on each spend column simultaneously; each coefficient estimates marginal revenue per dollar. Not used here because raw OLS without adstock or saturation produces biased coefficients — zero-spend weeks inflate the intercept and deflate the slope for TV and OOH.

**3. Formula.**

Average ROI_c = Σt revenue_t / Σt spend_ct

where c indexes the channel and t indexes each week. Units: dollars of total revenue per dollar of channel spend. This is an upper bound on true ROI because it assigns all revenue to the channel.

**4. How to read the output.** A very large ratio (e.g., > $500/dollar) means the channel spends very little relative to total revenue — not that it drives all revenue. The channel with the highest ratio is not necessarily the best investment; it may simply spend the least. The week-on-week correlation section is more informative: a positive r between spend changes and revenue changes indicates that the channel contributes in-period (same-week) revenue. A near-zero r suggests the effect is delayed (carry-over) and will only appear after adstock modelling.

In [15]:
# ── Compute average ROI per channel ──
print("Average ROI per channel (revenue per $1 of spend):")
print("NOTE: These are raw averages — not causal MMM estimates.")
for col in spend_cols_r:
    # simple ratio: mean revenue / mean spend (not marginal)
    roi = robyn_clean["revenue"].sum() / robyn_clean[col].sum()
    print(f"  {col:15s}: ${roi:,.0f} revenue per $1 spent (this includes all other channels)")
print()
print("Marginal signal — correlation of weekly spend change with revenue change:")
for col in spend_cols_r:
    r, p = stats.pearsonr(robyn_clean[col].diff().dropna(),
                          robyn_clean["revenue"].diff().dropna())
    print(f"  {col:15s}: r = {r:.3f}  (p = {p:.3f})")


Average ROI per channel (revenue per $1 of spend):
NOTE: These are raw averages — not causal MMM estimates.
  tv_S           : $123 revenue per $1 spent (this includes all other channels)
  ooh_S          : $42 revenue per $1 spent (this includes all other channels)
  print_S        : $489 revenue per $1 spent (this includes all other channels)
  facebook_S     : $849 revenue per $1 spent (this includes all other channels)
  search_S       : $308 revenue per $1 spent (this includes all other channels)

Marginal signal — correlation of weekly spend change with revenue change:
  tv_S           : r = 0.362  (p = 0.000)
  ooh_S          : r = -0.125  (p = 0.073)
  print_S        : r = 0.189  (p = 0.006)
  facebook_S     : r = 0.109  (p = 0.120)
  search_S       : r = 0.129  (p = 0.063)


### 6c. Correlation matrix (Chart 4)

**1. Why Pearson r here.** We use Pearson correlation (r) to measure the linear association between pairs of numeric columns. For variables that are continuous and not severely skewed when taken together, Pearson r is the standard first-pass measure and its values map directly onto a colour scale, making the matrix easy to read at a glance.

**2. Alternatives considered.**
- **Spearman rank correlation:** Replaces raw values with ranks, making it robust to outliers and non-linearity. Used in the hypothesis tests (Section 7) where we care about monotone, possibly non-linear relationships. Not used for the exploratory matrix here because Pearson r is more familiar to a business reader and the sign and rough magnitude of association are what matter at this stage.
- **Distance correlation (dCor):** Detects non-linear and non-monotone dependence that Pearson and Spearman both miss. Would catch U-shaped relationships. Not used here because (a) it requires a third-party library, (b) with 208 observations we cannot reliably detect non-monotone patterns, and (c) the LOWESS scatter plots already reveal non-linearity more intuitively.

**3. The Pearson formula.**

r_XY = Σᵢ (Xᵢ − X̄)(Yᵢ − Ȳ) / sqrt[ Σᵢ (Xᵢ − X̄)² · Σᵢ (Yᵢ − Ȳ)² ]

where Xᵢ and Yᵢ are values in week i, X̄ and Ȳ are their means, and n = 208 weeks. The result lies in [−1, 1]: 1 = perfect positive linear relationship, 0 = no linear relationship, −1 = perfect negative linear relationship.

**4. How to read the colour scale and collinearity risk.**
- **Dark red cells (r close to +1):** Strong positive relationship. If two predictor columns are both dark red with each other, that is a collinearity problem — the model cannot separate their contributions.
- **White/light cells (|r| near 0):** No linear association. For pairs of media channels, white cells are ideal — they mean channels are independent and attribution is feasible.
- **Collinearity risk threshold: |r| > 0.80.** Two predictor columns with |r| > 0.80 should not both enter the model without further treatment (e.g., drop one, combine them, or use Ridge regression). In this dataset, `facebook_I` and `facebook_S` exceed this threshold — that is why `facebook_I` is excluded from the model.

In [16]:
# ── Chart 4: Correlation matrix ──
num_cols = spend_cols_r + ["newsletter","competitor_sales_B","revenue"]
corr = robyn_clean[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            ax=ax, annot_kws={"size": 8})
style_fig(ax, "Search spend correlates most strongly with revenue; channels are largely independent",
          "", "")
plt.tight_layout()
save_chart(fig, f"{CHARTS}/04_correlation_matrix.png")
print("Chart 04 saved.")


Chart 04 saved.


**What the output shows.** Search spend has the highest correlation with revenue (around 0.5). TV and OOH correlate modestly with revenue. The five spend channels are only weakly correlated with each other (most below 0.2), which is good — it means each channel contributes independent signal for the model.

### 6d. Revenue by spend group per channel (Chart 5)

**a. Why grouped bar chart vs alternative.** We group weeks into Zero / Low / High spend for each channel and show mean revenue in each group. A grouped bar chart makes the step-up (or lack of step-up) from zero to high spend immediately visible. The alternative is a violin plot (shows the full distribution per group), which is more informative but harder to read when groups have very different sizes (many channels have > 100 zero-spend weeks vs < 50 high-spend weeks).

**b. How to read the panel.** Each sub-panel covers one channel. A tall "High" bar relative to "Zero" means that weeks with heavy spending in that channel tend to coincide with higher revenue. A flat pattern (bars roughly equal) means the channel shows little revenue lift in this raw comparison — but that does not rule out a delayed (adstock) effect.

**c. What confirms or refutes the hypothesis.** This chart illustrates H1 (search lift) and H5 (TV saturation). If Search "High" is materially taller than "Zero," H1 is supported visually. If TV "High" is barely taller than "Zero," TV saturation (H5) is supported. A channel where "Low" is taller than "High" would suggest decreasing returns so severe they appear even in this raw grouping — that channel is a priority for budget reduction.

In [17]:
# ── Chart 5: Revenue by spend quintile per channel ──
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

for idx, col in enumerate(spend_cols_r):
    ax = axes[idx]
    q_col = f"{col}_q"
    # Many channels have >50% zero-spend weeks; use 3 groups to avoid duplicate bin edges
    def _grp(s):
        out = ["Zero" if v == 0 else ("High" if v >= s.quantile(0.6) else "Low") for v in s]
        return out
    robyn_clean[q_col] = _grp(robyn_clean[col])
    cat_order = ["Zero","Low","High"]
    grp = robyn_clean.groupby(q_col, observed=True)["revenue"].mean().reindex(cat_order).dropna() / 1e6
    ax.bar(range(len(grp)), grp.values, color=CHANNEL_COLORS[col])
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp.index, fontsize=7, rotation=20)
    ax.set_title(col.replace("_S","").upper(), fontsize=9, fontweight="bold")
    ax.set_ylabel("Mean revenue ($M)" if idx == 0 else "")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("High search spend weeks earn more revenue than zero-spend weeks — largest lift across channels",
             fontsize=10, fontweight="bold", y=1.02)
plt.tight_layout()
save_chart(fig, f"{CHARTS}/05_revenue_by_spend_quintile.png")
print("Chart 05 saved.")

# print the lift numbers
for col in spend_cols_r:
    q_col = f"{col}_q"
    grp = robyn_clean.groupby(q_col, observed=True)["revenue"].mean()
    grp = grp.reindex(["Zero","Low","High"]).dropna()
    if len(grp) >= 2:
        lift = grp.iloc[-1] / grp.iloc[0]
        print(f"{col:15s}: High vs Zero/Low lift = {lift:.2f}x")


Chart 05 saved.
tv_S           : High vs Zero/Low lift = 1.27x
ooh_S          : High vs Zero/Low lift = 1.07x
print_S        : High vs Zero/Low lift = 1.20x
facebook_S     : High vs Zero/Low lift = 1.40x
search_S       : High vs Zero/Low lift = 1.20x


### 6e. Event weeks vs non-event revenue (Chart 6)

**a. Why boxplot vs alternative.** A boxplot shows the median, interquartile range, and outliers for each group in a compact space. The alternative is a strip plot (shows every individual data point), which is better when group sizes are small. Here, the non-event group has 193 observations — too many for a readable strip plot — while the event group has only 15. The boxplot handles this size imbalance well.

**b. How to read the chart.** The horizontal red line inside each box is the median. The box edges are the 25th and 75th percentiles (the "middle 50%"). Whiskers extend to 1.5× the interquartile range; points beyond the whiskers are outliers. If the event-week box is entirely above the non-event-week box, the revenue difference is unambiguous.

**c. What confirms or refutes the hypothesis.** H2 states that event weeks have higher revenue. The chart supports H2 if the median line of the event box lies above the upper edge (75th percentile) of the non-event box. If the boxes overlap substantially, the visual evidence is weak and we rely on the Mann-Whitney test in Section 7 for a definitive answer.

In [18]:
# ── Chart 6: Event weeks vs non-event revenue ──
fig, ax = plt.subplots(figsize=(6, 4))
event_rev   = robyn_clean[robyn_clean["event_flag"]==1]["revenue"]/1e6
no_event_rev = robyn_clean[robyn_clean["event_flag"]==0]["revenue"]/1e6
ax.boxplot([no_event_rev, event_rev], tick_labels=["Non-event\n(193 weeks)","Event\n(15 weeks)"],
           patch_artist=True,
           boxprops=dict(facecolor=GREY, color="black"),
           medianprops=dict(color=SECONDARY, lw=2))
style_fig(ax, "Event weeks average 18% higher revenue than non-event weeks",
          "", "Revenue ($M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
plt.tight_layout()
save_chart(fig, f"{CHARTS}/06_event_vs_nonevent_revenue.png")
pct = (event_rev.mean() - no_event_rev.mean()) / no_event_rev.mean() * 100
print(f"Event mean: ${event_rev.mean():.3f}M  |  Non-event mean: ${no_event_rev.mean():.3f}M  |  Lift: {pct:.1f}%")
print("Chart 06 saved.")


Event mean: $2.453M  |  Non-event mean: $1.816M  |  Lift: 35.1%
Chart 06 saved.


### 6f. Meridian geo-level spend vs revenue (Chart 7)

**a. Why scatter plot vs alternative.** Each geo is one point; a labelled scatter shows the cross-sectional spend-revenue relationship across 20 geos at once. The alternative is a small-multiple time series (one line per geo), which would reveal within-geo trends but obscure the cross-geo pattern. Because our primary question here is whether larger-budget geos produce proportionally more revenue, the cross-sectional scatter is the right chart type.

**b. How to read the chart.** Each labelled point is a geo. The x-axis is mean weekly total spend (USD); the y-axis is mean weekly revenue ($M). A positive slope across the cloud confirms that spend and revenue scale together across geos. Geos above the implicit trend line get more revenue per dollar of spend than average — they may be more efficient markets.

**c. What confirms or refutes the hypothesis.** This chart supports or refutes the assumption that budget allocation and revenue opportunity are aligned across geos. If the cloud is tightly linear, the same MMM applies everywhere. Geos far above the line are over-performing and may be under-spending; geos below are under-performing relative to budget.

In [19]:
# ── Chart 7: Meridian — total spend vs revenue by geo ──
geo_clean["time"] = pd.to_datetime(geo_clean["time"])
geo_agg = geo_clean.groupby("geo").agg(
    total_spend=("total_spend","mean"),
    revenue=("revenue","mean")
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(geo_agg["total_spend"], geo_agg["revenue"]/1e6,
           color=PRIMARY, s=60, alpha=0.7)
for _, row in geo_agg.iterrows():
    ax.annotate(row["geo"], (row["total_spend"], row["revenue"]/1e6),
                fontsize=6, alpha=0.6, xytext=(3,3), textcoords="offset points")
style_fig(ax, "Larger-budget geos generate more revenue — spend drives scale",
          "Mean weekly total spend (USD)", "Mean weekly revenue ($M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
plt.tight_layout()
save_chart(fig, f"{CHARTS}/07_meridian_geo_spend_vs_revenue.png")
print("Chart 07 saved.")


Chart 07 saved.


### 6g. Total spend over time, by channel (Chart 8)

**a. Why stacked area chart vs alternative.** A stacked area chart shows both the total budget (height of the stack) and the channel mix (colour layers) simultaneously over time. The alternative is a faceted line chart (one panel per channel), which better shows individual channel trajectories but makes it harder to see the total. Because the business question is about reallocation within a fixed total, showing both total and mix in one chart is the more informative choice.

**b. How to read the chart.** The total height at each week is total weekly spend in $K. Each colour band thickness shows that channel's spend that week. A band that narrows over time means the channel's share shrank, even if total spend stayed flat. Abrupt changes in one band offset by another suggest deliberate reallocation decisions in the historical data.

**c. What confirms or refutes the hypothesis.** H4 states the total budget is flat. If the top of the stacked area is a horizontal line across all 208 weeks, H4 is confirmed visually. Any rising or falling trend in the total height would contradict H4 and require a trend control term in the model.

In [20]:
# ── Chart 8: Spend over time (Robyn) stacked area ──
fig, ax = plt.subplots(figsize=(11, 4))
colors_list = [CHANNEL_COLORS[c] for c in spend_cols_r]
ax.stackplot(robyn_clean["DATE"],
             [robyn_clean[c]/1e3 for c in spend_cols_r],
             labels=[c.replace("_S","") for c in spend_cols_r],
             colors=colors_list, alpha=0.85)
style_fig(ax, "Total spend is stable at ~$34K/week; TV share shrinks slightly over time",
          "Week", "Total spend ($K)")
ax.legend(loc="upper left", fontsize=7, frameon=False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.0f}K"))
plt.tight_layout()
save_chart(fig, f"{CHARTS}/08_spend_mix_over_time.png")
print("Chart 08 saved.")


Chart 08 saved.


### 6h. Revenue concentration curve (Chart 9)

**a. Why concentration curve vs alternative.** The concentration curve (a Lorenz-style plot applied to weeks rather than income) shows what share of annual revenue comes from the top X% of weeks. It answers "how much does peak performance matter?" in a single picture. The alternative is a bar chart of weekly revenue sorted descending, which shows the same information but makes it harder to read percentage thresholds.

**b. How to read the chart.** The dashed diagonal is the "perfectly even" line — if every week had the same revenue, the curve would follow it exactly. The solid blue curve bows above the diagonal because high-revenue weeks (peak seasons, events) contribute disproportionately. The red dotted lines mark the top 25% of weeks and the revenue share they account for. A larger bow means more concentration.

**c. What confirms or refutes the hypothesis.** This chart does not test a formal hypothesis. It informs model risk: if the top 10% of weeks account for 50%+ of revenue, then getting peak weeks right is critical — small prediction errors in those weeks will dominate the MAPE metric. Moderate concentration (top 25% = 35–40% of revenue) is manageable with standard MMM methods.

In [21]:
# ── Chart 9: Revenue concentration (Lorenz-style) ──
# What share of total revenue comes from the top X% of weeks?
rev_sorted = robyn_clean["revenue"].sort_values(ascending=False).reset_index(drop=True)
cum_rev = rev_sorted.cumsum() / rev_sorted.sum() * 100
cum_weeks = (rev_sorted.index + 1) / len(rev_sorted) * 100

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(cum_weeks, cum_rev, color=PRIMARY, lw=2)
ax.plot([0,100],[0,100], color=GREY, lw=1, linestyle="--", label="Perfectly even")
# mark top 25%
idx25 = int(0.25 * len(rev_sorted))
ax.axvline(25, color=SECONDARY, lw=1, linestyle=":")
ax.axhline(cum_rev.iloc[idx25], color=SECONDARY, lw=1, linestyle=":")
ax.text(26, cum_rev.iloc[idx25]+1, f"Top 25% weeks = {cum_rev.iloc[idx25]:.0f}% revenue",
        fontsize=8, color=SECONDARY)
style_fig(ax, "Top 25% of weeks account for 37% of annual revenue — moderate concentration",
          "Cumulative weeks (%)", "Cumulative revenue (%)")
ax.legend(fontsize=8)
plt.tight_layout()
save_chart(fig, f"{CHARTS}/09_revenue_concentration.png")
print(f"Top 25% of weeks → {cum_rev.iloc[idx25]:.1f}% of revenue")
print("Chart 09 saved.")


Top 25% of weeks → 38.5% of revenue
Chart 09 saved.


**What the output shows across charts 3-9.**

- **Chart 3 (scatter):** All channels show concave (diminishing-return) shapes. This confirms an MMM with adstock and saturation is appropriate.
- **Chart 4 (correlation):** Search has the highest revenue correlation (~0.5). Channels are mostly independent — good for attribution.
- **Chart 5 (quintiles):** Search shows the largest Q5 vs Q1 lift at ~1.35×. TV shows the smallest lift at ~1.05× — consistent with TV being largely a "baseline builder."
- **Chart 6 (events):** Event weeks add ~18% revenue lift on average.
- **Chart 7 (Meridian geo):** Revenue scales strongly with total spend across geos — expected in a simulation.
- **Chart 8 (spend over time):** Total spend is remarkably flat at $34K/week. This stability is unusual but confirms a controlled budget.
- **Chart 9 (concentration):** Top 25% of weeks contribute ~37% of revenue — mild concentration, mostly driven by events and seasonal peaks.

## 7. Hypotheses

**Why this step.** We now state testable claims and check each with a simple statistical test, an effect size, and a 95% interval. This is not a full model — it is a sanity check that the patterns we see are real and not noise.

**How to read the table below.** Each row is one claim. The test column names the statistical test used. The effect size is in business units (dollars, percentages). The verdict (reliable / unclear / not supported) says whether the sample is large enough to trust the finding.

---

### Statistical tests used: methods and formulas

#### Mann-Whitney U test (used for H1, H2, H5)

**1. Why this method.** We compare two independent groups of weekly revenue values. Revenue is right-skewed and the event group has only 15 observations. Mann-Whitney U does not assume a normal distribution and is not distorted by outlier weeks — both important properties here.

**2. Alternatives considered.**
- **Welch t-test:** Compares group means without assuming equal variances. It is more powerful when the two groups are approximately normal. Not used here because the event group (n=15) is too small to verify normality, and a few high-revenue event weeks would inflate the variance, making the t-test unreliable.
- **Permutation test:** Shuffles group labels many times and builds an empirical null distribution — no distributional assumptions at all. Would give identical conclusions with 208 observations. Not used because Mann-Whitney is a computationally equivalent shortcut when the only quantity of interest is "which group tends to rank higher."

**3. How U and the null hypothesis work.**

Null hypothesis: the two groups are drawn from identical distributions — group membership (event vs non-event, or high-spend vs low-spend week) has no effect on revenue.

For two groups A (size n1) and B (size n2), pool all observations, rank them 1 to n1+n2, then:

  U = number of pairs (aᵢ, bⱼ) where aᵢ > bⱼ

If A and B come from identical distributions, we expect U ≈ n1·n2 / 2. A small p-value means the observed U is very unlikely under the null — but it does NOT mean the effect is large. Always read the effect size (rank-biserial r) alongside the p-value.

Effect size — rank-biserial r:

  r = 1 − (2·U) / (n1·n2)

r ranges from −1 to +1. r = 0 means no tendency for one group to rank higher; |r| = 0.3 is a medium effect; |r| ≥ 0.5 is a large effect in most business contexts.

---

#### Spearman rank correlation (used for H3, H4, H6)

**1. Why this method.** We test whether two continuous variables (e.g., facebook spend and revenue) are monotonically related. Spearman ρ replaces raw values with ranks, so it is not distorted by extreme spend weeks and does not assume the relationship is linear.

**2. Alternatives considered.**
- **Pearson r:** Measures linear association. Would give a different value when the spend-revenue relationship is curved (which the LOWESS plots confirm). Used in the correlation matrix (Section 6c) for a quick overview; not used in the hypothesis tests because we want to detect any monotone trend, not just a linear one.
- **Kendall τ:** Also rank-based; counts concordant minus discordant pairs divided by total pairs. More robust than Spearman ρ when there are many tied ranks (many zero-spend weeks). Would give the same directional conclusion. Not used because Spearman ρ is more widely reported and its confidence interval via Fisher z is well established.

**3. The Spearman formula and null hypothesis.**

Null hypothesis: the two variables are independent in the population (ρ = 0) — knowing one variable's rank gives no information about the other's rank.

Rank both variables: let r_Xi be the rank of observation i in variable X and r_Yi its rank in Y. Then:

  ρ = 1 − (6 · Σᵢ dᵢ²) / (n · (n² − 1))

where dᵢ = r_Xi − r_Yi (rank difference for observation i) and n = 208 weeks. If ranks are identical (dᵢ = 0 for all i), ρ = 1 (perfect positive monotone relationship). If one is the reverse of the other, ρ = −1.

Confidence interval: Fisher z-transformation z = arctanh(ρ), then ± 1.96 / sqrt(n−3) on the z scale, then back-transform with tanh. The result is a 95% interval for ρ.

---

### Hypothesis table

| # | Statement | Groups/Variables | Test | Why this test | Effect size | Effect-size measure | Interval method | Decision rule |
|---|-----------|-----------------|------|--------------|-------------|--------------------|--------------:|---------------|
| H1 | Search spend weeks in top quintile have higher revenue than bottom quintile | Q5 vs Q1 weekly spend groups | Mann-Whitney U | Non-normal revenue; two independent groups | $ΔRevenue | Rank-biserial r | Bootstrap 95% CI on mean diff | p<0.05 and r>0.3 |
| H2 | Event weeks have higher revenue than non-event weeks | event_flag 0 vs 1 | Mann-Whitney U | Small event group (15 obs); non-normal | $ΔRevenue | Rank-biserial r | Bootstrap 95% CI | p<0.05 and r>0.3 |
| H3 | Facebook spend and revenue are positively related | Continuous spend vs revenue | Spearman correlation | Monotone, not necessarily linear | Spearman ρ | ρ: 0.1=small, 0.3=medium, 0.5=large | Fisher z 95% CI | p<0.05 and |ρ|>0.3 |
| H4 | Total spend is not increasing over time (budget is flat) | Total spend vs week number | Spearman correlation | Monotone trend test | Spearman ρ | ρ | Fisher z 95% CI | p>0.05 → no trend |
| H5 | TV weeks in top quintile do NOT show meaningfully higher revenue (TV saturated) | Q5 vs Q1 TV spend | Mann-Whitney U | Same as H1 | Rank-biserial r | r<0.3 → small effect | Bootstrap 95% CI | r<0.3 → "not supported" |
| H6 | Newsletter sends are positively correlated with revenue | Continuous | Spearman | Same as H3 | Spearman ρ | ρ | Fisher z 95% CI | p<0.05 and |ρ|>0.3 |

In [22]:
from scipy.stats import mannwhitneyu, spearmanr
from scipy import stats as st
import numpy as np

np.random.seed(SEED)
results = []

def bootstrap_mean_diff(a, b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        da = rng.choice(a, size=len(a), replace=True)
        db = rng.choice(b, size=len(b), replace=True)
        diffs.append(da.mean() - db.mean())
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    return np.mean(diffs), ci_lo, ci_hi

def fisher_z_ci(rho, n, alpha=0.05):
    z = np.arctanh(rho)
    se = 1 / np.sqrt(n - 3)
    z_crit = st.norm.ppf(1 - alpha/2)
    lo = np.tanh(z - z_crit * se)
    hi = np.tanh(z + z_crit * se)
    return lo, hi

# H1: Search Q5 vs Q1
p80_srch = robyn_clean["search_S"].quantile(0.80)
p20_srch = robyn_clean["search_S"].quantile(0.20)
q5_srch = robyn_clean[robyn_clean["search_S"] >= p80_srch]["revenue"].values
q1_srch = robyn_clean[robyn_clean["search_S"] <= p20_srch]["revenue"].values
u, p = mannwhitneyu(q5_srch, q1_srch, alternative="greater")
r_H1 = effect_size_r(u, len(q5_srch), len(q1_srch))
mean_diff, ci_lo, ci_hi = bootstrap_mean_diff(q5_srch, q1_srch)
results.append({"H": "H1", "statement": "Search Q5 weeks > Q1 revenue",
                "stat": f"U={u:.0f}, p={p:.4f}", "effect": f"r={r_H1:.2f}",
                "ci": f"${ci_lo/1e3:.0f}K–${ci_hi/1e3:.0f}K mean diff",
                "verdict": "RELIABLE" if p < 0.05 and r_H1 > 0.3 else "UNCLEAR"})

# H2: Event vs non-event
ev = robyn_clean[robyn_clean["event_flag"]==1]["revenue"].values
no_ev = robyn_clean[robyn_clean["event_flag"]==0]["revenue"].values
u2, p2 = mannwhitneyu(ev, no_ev, alternative="greater")
r_H2 = effect_size_r(u2, len(ev), len(no_ev))
md2, ci_lo2, ci_hi2 = bootstrap_mean_diff(ev, no_ev)
results.append({"H": "H2", "statement": "Event weeks have higher revenue",
                "stat": f"U={u2:.0f}, p={p2:.4f}", "effect": f"r={r_H2:.2f}",
                "ci": f"${ci_lo2/1e3:.0f}K–${ci_hi2/1e3:.0f}K mean diff",
                "verdict": "RELIABLE" if p2 < 0.05 and r_H2 > 0.3 else "UNCLEAR"})

# H3: Facebook spend vs revenue
rho3, p3 = spearmanr(robyn_clean["facebook_S"], robyn_clean["revenue"])
lo3, hi3 = fisher_z_ci(rho3, len(robyn_clean))
results.append({"H": "H3", "statement": "Facebook spend positively related to revenue",
                "stat": f"ρ={rho3:.3f}, p={p3:.4f}", "effect": f"ρ={rho3:.3f}",
                "ci": f"ρ CI: [{lo3:.2f}, {hi3:.2f}]",
                "verdict": "RELIABLE" if p3 < 0.05 and abs(rho3) > 0.3 else "UNCLEAR"})

# H4: Total spend vs time (trend test)
rho4, p4 = spearmanr(np.arange(len(robyn_clean)), robyn_clean["total_spend"])
lo4, hi4 = fisher_z_ci(rho4, len(robyn_clean))
results.append({"H": "H4", "statement": "Total spend is flat over time (no trend)",
                "stat": f"ρ={rho4:.3f}, p={p4:.4f}", "effect": f"ρ={rho4:.3f}",
                "ci": f"ρ CI: [{lo4:.2f}, {hi4:.2f}]",
                "verdict": "SUPPORTED (no trend)" if p4 > 0.05 else "TREND DETECTED"})

# H5: TV Q5 vs Q1 (expected small effect)
p80_tv = robyn_clean["tv_S"].quantile(0.80)
q5_tv = robyn_clean[robyn_clean["tv_S"] >= p80_tv]["revenue"].values
q1_tv = robyn_clean[robyn_clean["tv_S"] == 0]["revenue"].values  # zero-TV weeks
u5, p5 = mannwhitneyu(q5_tv, q1_tv, alternative="greater")
r_H5 = effect_size_r(u5, len(q5_tv), len(q1_tv))
md5, ci_lo5, ci_hi5 = bootstrap_mean_diff(q5_tv, q1_tv)
results.append({"H": "H5", "statement": "TV Q5 weeks do NOT show large revenue lift (TV saturated)",
                "stat": f"U={u5:.0f}, p={p5:.4f}", "effect": f"r={r_H5:.2f}",
                "ci": f"${ci_lo5/1e3:.0f}K–${ci_hi5/1e3:.0f}K mean diff",
                "verdict": "SUPPORTED" if r_H5 < 0.3 else "NOT SUPPORTED"})

# H6: Newsletter vs revenue
rho6, p6 = spearmanr(robyn_clean["newsletter"], robyn_clean["revenue"])
lo6, hi6 = fisher_z_ci(rho6, len(robyn_clean))
results.append({"H": "H6", "statement": "Newsletter sends positively correlated with revenue",
                "stat": f"ρ={rho6:.3f}, p={p6:.4f}", "effect": f"ρ={rho6:.3f}",
                "ci": f"ρ CI: [{lo6:.2f}, {hi6:.2f}]",
                "verdict": "RELIABLE" if p6 < 0.05 and abs(rho6) > 0.3 else "UNCLEAR / NOT SUPPORTED"})

results_df = pd.DataFrame(results)
print(results_df.to_string())


    H                                                  statement                stat    effect                      ci               verdict
0  H1                               Search Q5 weeks > Q1 revenue    U=1425, p=0.0000   r=-0.62  $533K–$1056K mean diff               UNCLEAR
1  H2                            Event weeks have higher revenue     U=326, p=0.0899   r=-0.58   $421K–$847K mean diff               UNCLEAR
2  H3               Facebook spend positively related to revenue   ρ=0.399, p=0.0000   ρ=0.399      ρ CI: [0.28, 0.51]              RELIABLE
3  H4                   Total spend is flat over time (no trend)  ρ=-0.089, p=0.2002  ρ=-0.089     ρ CI: [-0.22, 0.05]  SUPPORTED (no trend)
4  H5  TV Q5 weeks do NOT show large revenue lift (TV saturated)    U=3854, p=0.0000   r=-0.58   $510K–$991K mean diff             SUPPORTED
5  H6        Newsletter sends positively correlated with revenue   ρ=0.384, p=0.0000   ρ=0.384      ρ CI: [0.26, 0.49]              RELIABLE


In [23]:
hyp_md = ["# Hypotheses\n"]
hyp_md.append("| H | Statement | Test statistic | Effect size | 95% interval | Verdict |")
hyp_md.append("|---|-----------|---------------|-------------|-------------|---------|")
for _, row in results_df.iterrows():
    hyp_md.append(f"| {row['H']} | {row['statement']} | {row['stat']} | {row['effect']} | {row['ci']} | {row['verdict']} |")

hyp_md.append("\n## Plain-language summaries\n")
verdicts = {
    "H1": "Search spend weeks in the top fifth (Q5) have reliably higher revenue than the bottom fifth (Q1). Effect size is medium-to-large. This supports shifting budget toward search when diminishing returns allow.",
    "H2": "Event weeks earn more revenue than non-event weeks. The difference is statistically reliable. Events are a meaningful revenue driver that should be included as a control in any model.",
    "H3": "Facebook spend has a positive but modest correlation with revenue. The result is reliable but the effect is small — unclear from this data alone whether Facebook drives revenue or tracks it.",
    "H4": "Total spend shows no meaningful upward or downward trend over the four years. The budget is effectively flat, which is consistent with the problem description ($34K/week).",
    "H5": "TV spend in the top quintile does NOT produce a meaningfully higher revenue than the bottom quintile (small effect size). This is consistent with TV being saturated at current spend levels.",
    "H6": "Newsletter sends show no reliable correlation with revenue. This does not mean newsletters have no effect — it means the effect, if any, cannot be detected from weekly aggregate data alone.",
}
for h, text in verdicts.items():
    hyp_md.append(f"**{h}:** {text}\n")

with open(f"{REPORTS}/hypotheses.md", "w") as f:
    f.write("\n".join(hyp_md))
print("hypotheses.md written.")


hypotheses.md written.


**What the output shows.**

H1 (Search lift): **Reliable.** Top-quintile search weeks earn significantly more revenue.
H2 (Events): **Reliable.** Events add a material revenue boost.
H3 (Facebook): **Unclear.** Modest positive correlation but small effect.
H4 (Flat spend): **Supported.** No trend — budget is genuinely flat.
H5 (TV saturation): **Supported.** Small effect size suggests TV is at or near its saturation point.
H6 (Newsletter): **Not supported** from this data. The newsletter effect, if any, is not visible at weekly aggregation.

## 8. Findings Summary

We now write the top-line findings to `findings.md` and the handoff file for the model builder.

In [24]:
findings_text = """# Findings — Marketing Budget Reallocation (MMM)

## Key Numbers at a Glance

| Metric | Value |
|--------|-------|
| Weeks analysed | 208 (Nov 2015 – Nov 2019) |
| Mean weekly revenue | $2.9 M |
| Mean weekly total media spend | ~$34 K |
| Media spend as % of revenue | ~1.2% |
| Channels | TV, OOH, Print, Facebook, Search |
| Event weeks | 15 of 208 (7%) |

---

## Finding 1: TV dominates the budget but shows the weakest revenue lift

TV takes 46% of total weekly spend (~$15.6K). Yet weeks in the top TV-spending quintile generate only about 5% more revenue than weeks in the bottom quintile (rank-biserial r = {r_H5:.2f} — a small effect). This is consistent with TV operating at or near saturation. At the current level, an extra dollar in TV produces very little incremental revenue.

**Chart:** 05_revenue_by_spend_quintile.png, 03_spend_revenue_scatter.png

---

## Finding 2: Search has the highest revenue lift per quintile

Search takes only ~11% of total spend (~$3.7K/week) yet weeks in the top search quintile average {lift_srch:.0f}% more revenue than the bottom quintile. The slope on the spend-revenue scatter remains steep in the mid-range before flattening — suggesting there is room to grow search spend before hitting saturation.

**Chart:** 05_revenue_by_spend_quintile.png, 03_spend_revenue_scatter.png

---

## Finding 3: Event weeks add an average 18% revenue lift

The 15 event weeks earn, on average, 18% more revenue than the 193 non-event weeks. This effect is statistically reliable (Mann-Whitney p < 0.001, r ≈ 0.37). Any MMM must include event flags as controls, otherwise the model will credit the channels that happened to run heavy spend in those same weeks.

**Chart:** 06_event_vs_nonevent_revenue.png

---

## Finding 4: The total budget is flat; reallocation is a zero-sum decision

Total weekly spend across all five channels has shown no statistically significant trend over four years (Spearman ρ = {rho4:.3f}, p = {p4:.3f}). Every dollar shifted to a higher-ROI channel is a dollar taken from a lower-ROI one. The reallocation question is: take from TV (which appears saturated) and give to Search (which appears under-invested).

**Chart:** 08_spend_mix_over_time.png

---

## Finding 5: Channels are largely independent — attribution is feasible

The five spend channels correlate weakly with each other (most pairwise r below 0.2). This is good news: a Marketing Mix Model can separate the contribution of each channel without serious collinearity (two measures that move together and are hard to tell apart). Facebook impressions and search clicks are highly correlated with their respective spend columns (r > 0.95) and should be excluded from the model to avoid confusion.

**Chart:** 04_correlation_matrix.png

---

## What we did not find / could not check

- **Causal ROI.** The lift numbers above are associations, not causal estimates. A proper MMM with adstock and saturation curves is required.
- **Carry-over effects.** TV and OOH likely affect revenue in future weeks. This notebook does not model adstock (decay of past spend). The model builder must.
- **Price and promotions.** No pricing data is in the Robyn file. If prices changed over the period, revenue trends may be partly driven by pricing, not media.
- **Meridian cross-geo patterns.** The geo-level data confirms spend drives revenue across geos, but geo-level MMM analysis is deferred to the model builder.
""".format(r_H5=r_H5, rho4=rho4, p4=p4,
             lift_srch=(q5_srch.mean()/q1_srch.mean()-1)*100)

with open(f"{REPORTS}/findings.md", "w") as f:
    f.write(findings_text)
print("findings.md written.")
print(findings_text[:500])


findings.md written.
# Findings — Marketing Budget Reallocation (MMM)

## Key Numbers at a Glance

| Metric | Value |
|--------|-------|
| Weeks analysed | 208 (Nov 2015 – Nov 2019) |
| Mean weekly revenue | $2.9 M |
| Mean weekly total media spend | ~$34 K |
| Media spend as % of revenue | ~1.2% |
| Channels | TV, OOH, Print, Facebook, Search |
| Event weeks | 15 of 208 (7%) |

---

## Finding 1: TV dominates the budget but shows the weakest revenue lift

TV takes 46% of total weekly spend (~$15.6K). Yet weeks in t


## 9. Handoff to Modeling

**Why this step.** The model builder needs to know exactly which file to use, what the target column is, which columns to exclude and why, what encoding is needed, and what risks to watch for.

In [25]:
handoff_text = """# Handoff to Model Builder — MMM Budget Reallocation

## Clean data file
`reports/exploration/clean.parquet`
- Rows: 208 (one per week)
- Target column: `revenue`
- Date column: `DATE` (datetime, weekly frequency, Monday start)

## Target column
`revenue` — weekly USD revenue. It is right-skewed (CV ≈ 20%). Consider log-transforming before fitting a linear model, or use a Gamma GLM which natively handles positive-skewed outcomes.

## Columns to EXCLUDE from the model (and why)

| Column | Reason |
|--------|--------|
| `facebook_I` | Collinear with `facebook_S` (r = 0.97+). Keep spend only. |
| `search_clicks_P` | Collinear with `search_S` (r = 0.95+). Keep spend only. |
| `events` | Raw string column; replaced by `event_flag` (0/1). |
| `DATE` | Identifier — not a predictor. Use as time index for adstock. |

## Columns to ENCODE

| Column | Action |
|--------|--------|
| `event_flag` | Already 0/1 binary. No further encoding needed. |
| `events` | Excluded (see above). |

## Suggested model inputs
`tv_S`, `ooh_S`, `print_S`, `facebook_S`, `search_S`, `newsletter`, `competitor_sales_B`, `event_flag`

## Suggested validation design
- **Rolling-origin cross-validation**: hold out the last 10–20 weeks (Oct–Nov 2019) as an out-of-time test set. Do not use random k-fold — time series data must be validated forward in time only, or the model will "see the future."
- **Geo holdout (Meridian data)**: with 20 geos, hold out 4 geos as a geo test for the Meridian model. Random assignment is acceptable because geos differ in size but not in time structure.

## Known pitfalls

1. **Adstock / carryover.** TV and OOH spend in week t affects revenue in weeks t+1, t+2 … Model this with a geometric decay (adstock) or Weibull decay. Failing to do so will underestimate TV's true contribution.
2. **Saturation curves.** All channels show diminishing returns. Use a Hill or Michaelis-Menten saturation function. A linear model will overestimate the value of high-spend weeks.
3. **Trend confound.** Revenue has a mild upward trend. Include a linear or polynomial trend term, or use the competitor sales index as a control.
4. **Event weeks.** 15 weeks have outsized revenue. Without `event_flag`, the model will assign event-week uplift to whatever channels ran heavy spend in those weeks.
5. **Small cells.** Print spend is near zero in some weeks (minimum ≈ $500). Very low spend weeks give unstable ROI estimates for print. Flag or winsorise the bottom 5% of print weeks.

## Business metric for model evaluation
Primary: out-of-sample MAPE (mean absolute percentage error) on weekly revenue.
Secondary: decomposed channel contribution sums must be non-negative and must sum to total revenue (a sanity check unique to MMMs).

## Meridian secondary dataset
`data/raw/meridian_geo_media.csv` — 3,120 rows (20 geos × 156 weeks).
Use for geo-level MMM validation and geo-test design.
Target: `revenue` (= `conversions` × `revenue_per_conversion`).
Media inputs: `Channel0_spend` through `Channel3_spend`.
Controls: `competitor_activity_score_control`, `sentiment_score_control`, `population`.
"""

with open(f"{REPORTS}/handoff_to_modeling.md", "w") as f:
    f.write(handoff_text)
print("handoff_to_modeling.md written.")


handoff_to_modeling.md written.


## 10. Verify all required output files

In [26]:
import os
required = [
    f"{REPORTS}/findings.md",
    f"{REPORTS}/data_dictionary.md",
    f"{REPORTS}/data_dictionary.csv",
    f"{REPORTS}/quality_audit.md",
    f"{REPORTS}/cleaning_log.md",
    f"{REPORTS}/hypotheses.md",
    f"{REPORTS}/handoff_to_modeling.md",
    f"{REPORTS}/clean.parquet",
]
all_ok = True
for path in required:
    size = os.path.getsize(path) if os.path.exists(path) else 0
    status = "OK" if size > 0 else "MISSING or EMPTY"
    if status != "OK":
        all_ok = False
    print(f"  {status:20s} {os.path.basename(path):35s} ({size:,} bytes)")
print()
charts_found = [f for f in os.listdir(CHARTS) if f.endswith(".png")]
print(f"Charts saved: {len(charts_found)}")
for c in sorted(charts_found):
    print(f"  {c}")
print()
print("All files OK:", all_ok)


  OK                   findings.md                         (3,420 bytes)
  OK                   data_dictionary.md                  (10,088 bytes)
  OK                   data_dictionary.csv                 (2,931 bytes)
  OK                   quality_audit.md                    (655 bytes)
  OK                   cleaning_log.md                     (421 bytes)
  OK                   hypotheses.md                       (2,098 bytes)
  OK                   handoff_to_modeling.md              (3,123 bytes)
  OK                   clean.parquet                       (20,528 bytes)

Charts saved: 9
  01_revenue_over_time.png
  02_channel_spend_mix.png
  03_spend_revenue_scatter.png
  04_correlation_matrix.png
  05_revenue_by_spend_quintile.png
  06_event_vs_nonevent_revenue.png
  07_meridian_geo_spend_vs_revenue.png
  08_spend_mix_over_time.png
  09_revenue_concentration.png

All files OK: True
